# MLflow 3 GenAI Evaluation: End-to-End Walkthrough

An opinionated, runnable tour of the evaluation lifecycle in Databricks, matching the
MLflow UI flow:

> **Trace → Sessions → Judges → Evaluation Datasets → Evaluation Runs → Labeling Schemas → Labeling Sessions → Prompts / Agent Versioning → Prompt Optimization → Production Monitoring**

The demo agent is a LangGraph ReAct research assistant that answers questions using
two tools against the public arXiv API. We deliberately build a weak v1, let
evaluation expose a hallucinated-citation failure, collect human labels on the
failures, ship a manual v2 via Prompt Registry, then auto-optimize a v3 with
`mlflow.genai.optimize_prompt`, and finally enable production monitoring with the
same scorers.

## What you'll learn

- Autologging a LangGraph agent and adding custom, named `SpanType.TOOL` spans
- Grouping multi-turn traces with session metadata
- Writing custom `@scorer`s that read **tool spans**: citation correctness, result counts, BLEU-1 between the question and retrieved abstracts
- Curating a Unity Catalog evaluation dataset with ground-truth expectations (no guidelines — those live in a `Guidelines` scorer)
- Running `mlflow.genai.evaluate()` and comparing versions via `run.data.metrics`
- Creating labeling schemas + a labeling session for reviewer feedback
- Versioning prompts and agents, and promoting via aliases
- Auto-evolving prompts with `mlflow.genai.optimize_prompt` against the same scorers
- Enabling the same scorers as **production monitors** via `scorer.register().start()`

## Prerequisites

- Databricks workspace with Foundation Model API access
- A Unity Catalog `catalog.schema` you can write to (Prompt Registry + eval dataset)
- Internet egress to `export.arxiv.org` (the only external dependency; no auth)

In [ ]:
%pip install -q 'mlflow[databricks]>=3.10' databricks-agents databricks-langchain langgraph langchain-core feedparser

In [ ]:
dbutils.library.restartPython()

## 0. Configuration

Set a Unity Catalog `CATALOG.SCHEMA` you can write to. Everything the notebook
creates — prompts, the evaluation dataset, and the labeling session — lands there.

In [ ]:
import os
import sys
import time

# === EDIT THESE ===
CATALOG = "shm"
SCHEMA = "genai"
# ==================

AGENT_ENDPOINT = "databricks-gpt-5"
JUDGE_ENDPOINT = "databricks-gpt-5"
EXPERIMENT_NAME = "arxiv_eval_walkthrough"

USER_EMAIL = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
os.environ["DATABRICKS_HOST"] = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiUrl().get()
)
os.environ["DATABRICKS_TOKEN"] = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().apiToken().get()
)

os.environ["OPENAI_API_KEY"] = os.environ["DATABRICKS_TOKEN"]
os.environ["OPENAI_API_BASE"] = f"{os.environ['DATABRICKS_HOST']}/serving-endpoints"

PROMPT_FQN = f"{CATALOG}.{SCHEMA}.arxiv_agent_system_prompt"
DATASET_FQN = f"{CATALOG}.{SCHEMA}.arxiv_eval_questions"

NB_DIR = os.path.dirname(os.path.abspath("arxiv_tools.py")) if os.path.exists("arxiv_tools.py") else os.getcwd()
if NB_DIR not in sys.path:
    sys.path.insert(0, NB_DIR)

print(f"User:       {USER_EMAIL}")
print(f"Prompt:     {PROMPT_FQN}")
print(f"Dataset:    {DATASET_FQN}")
print(f"Experiment: /Users/{USER_EMAIL}/{EXPERIMENT_NAME}")

In [ ]:
import mlflow

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(f"/Users/{USER_EMAIL}/{EXPERIMENT_NAME}")
mlflow.langchain.autolog()

experiment = mlflow.get_experiment_by_name(f"/Users/{USER_EMAIL}/{EXPERIMENT_NAME}")
EXPERIMENT_ID = experiment.experiment_id

print("MLflow:", mlflow.__version__)
print("Experiment ID:", EXPERIMENT_ID)

### Idempotent cleanup

Delete prior traces from this experiment so each run starts clean. We keep the
experiment itself (it may contain other runs you care about).

In [ ]:
# --- Hard cleanup for idempotent reruns ---
# Traces
try:
    now_ms = int(time.time() * 1000)
    total = 0
    for _ in range(20):  # up to 20k traces
        n = mlflow.delete_traces(
            experiment_id=EXPERIMENT_ID,
            max_timestamp_millis=now_ms,
            max_traces=1000,
        )
        if not n:
            break
        total += n
    print(f"Deleted {total} prior traces")
except Exception as e:
    print(f"Trace cleanup skipped: {e}")

# Dataset UC table (schema may have stale columns like old-style `guidelines`)
try:
    spark.sql(f"DROP TABLE IF EXISTS {DATASET_FQN}")
    print(f"Dropped table {DATASET_FQN}")
except Exception as e:
    print(f"Table drop skipped: {e}")

# Prompt + all versions (so we restart from v1)
try:
    for v in range(1, 20):
        try:
            mlflow.genai.delete_prompt_version(name=PROMPT_FQN, version=v)
        except Exception:
            pass
    try:
        mlflow.genai.delete_prompt(name=PROMPT_FQN)
        print(f"Deleted prompt {PROMPT_FQN}")
    except Exception as e:
        print(f"delete_prompt skipped: {type(e).__name__}: {e}")
except Exception as e:
    print(f"Prompt cleanup skipped: {e}")

# Labeling sessions (so they stop pinning label schemas)
try:
    from mlflow.genai import labeling
    for sess in labeling.get_labeling_sessions():
        try:
            labeling.delete_labeling_session(sess)
            print(f"Deleted labeling session: {sess.name}")
        except Exception as e:
            print(f"Could not delete session {sess.name}: {e}")
except Exception as e:
    print(f"Labeling session cleanup skipped: {e}")

# Label schemas (now unpinned)
try:
    from mlflow.genai import label_schemas
    for name in ("arxiv_answer_quality", "arxiv_cited_correctly", "arxiv_reviewer_notes"):
        try:
            label_schemas.delete_label_schema(name=name)
            print(f"Deleted label schema: {name}")
        except Exception:
            pass
except Exception as e:
    print(f"Label schema cleanup skipped: {e}")


## 1. Trace — instrument first, everything else builds on this

`mlflow.langchain.autolog()` traces the agent graph and LLM calls automatically.
On top of that, the two arXiv functions in `arxiv_tools.py` are decorated with
`@mlflow.trace(span_type=SpanType.TOOL, name=...)` so each tool invocation produces
a **named** TOOL span we can inspect from judges.

Below, we call each tool directly once. Open the trace in the MLflow UI and confirm
you see two TOOL spans: `search_arxiv` and `fetch_arxiv_paper`.

In [ ]:
from arxiv_tools import search_arxiv, fetch_arxiv_paper

with mlflow.start_span(name="tool_smoke_test") as span:
    results = search_arxiv("attention is all you need", max_results=3)
    first = fetch_arxiv_paper(results[0]["arxiv_id"])
    span.set_outputs({"first_title": first["title"]})

for r in results:
    print(f"[{r['arxiv_id']}] {r['title']}")
print()
print("Full metadata for top hit:")
print(first)

## 2. Sessions — group multi-turn traces by conversation

Setting the `mlflow.trace.session` metadata on a trace tells the MLflow UI to bucket
traces into sessions. `arxiv_agent.run_turn()` does this for us on every invocation.

First we need an agent. The system prompt lives in the UC Prompt Registry so we can
version it independently of the code. Register v1 now.

In [ ]:
SYSTEM_PROMPT_V1 = (
    "You are a research assistant. Answer the user's question using the arXiv tools "
    "available to you. Be concise."
)

mlflow.genai.register_prompt(
    name=PROMPT_FQN,
    template=SYSTEM_PROMPT_V1,
    commit_message="v1: thin prompt, no citation discipline",
)
mlflow.genai.set_prompt_alias(name=PROMPT_FQN, alias="production", version=1)
mlflow.genai.set_prompt_alias(name=PROMPT_FQN, alias="candidate", version=1)
print(f"Registered {PROMPT_FQN} v1 and set aliases @production, @candidate -> 1")

In [ ]:
from arxiv_agent import build_agent, load_prompt_by_alias, run_turn

# Tie v1 traces to a versioned LoggedModel entry
mlflow.set_active_model(name="arxiv-agent-v1")
mlflow.log_model_params({
    "prompt_fqn": PROMPT_FQN,
    "prompt_version": 1,
    "prompt_alias": "candidate",
    "llm_endpoint": AGENT_ENDPOINT,
    "tools": "search_arxiv,fetch_arxiv_paper",
})

system_prompt = load_prompt_by_alias(PROMPT_FQN, "candidate")
agent_v1 = build_agent(endpoint=AGENT_ENDPOINT, system_prompt=system_prompt)

print(agent_v1.get_graph().draw_mermaid())


In [ ]:
session_id = "walkthrough-demo-session-01"

with mlflow.start_run(run_name="multi-turn-demo"):
    reply_1 = run_turn(agent_v1, "What paper introduced the Transformer?", session_id=session_id, user=USER_EMAIL)
    print("Turn 1:\n", reply_1, "\n")
    reply_2 = run_turn(agent_v1, "Who were the authors of that paper?", session_id=session_id, user=USER_EMAIL)
    print("Turn 2:\n", reply_2)

## 3. Judges — built-in scorers + custom scorers that read tool spans

We combine six scorers:

- **`RelevanceToQuery`** and **`Safety`** — built-in LLM judges.
- **`Guidelines`** — built-in LLM judge that checks the answer against global rules
  (here: citation discipline). Guidelines go in a *scorer*, not in the dataset's
  expectations.
- **`citation_correctness`** — custom, deterministic. Walks the trace's TOOL spans,
  collects every `arxiv_id` the tools returned, and verifies every `[arxiv:ID]` in
  the answer was actually observed.
- **`num_results_returned`** — custom, quantitative. Counts how many rows each
  `search_arxiv` TOOL span returned.
- **`query_abstract_bleu`** — custom, quantitative. BLEU-1 unigram precision with
  brevity penalty between the user's question (candidate) and the abstracts the tools
  pulled back (reference). Higher means retrieval is lexically on-topic.

All six can be enabled as production monitors — see the last section.

In [ ]:
import json
import math
import re

from mlflow.entities import Feedback
from mlflow.genai import scorer
from mlflow.genai.scorers import Guidelines, RelevanceToQuery, Safety

CITATION_PATTERN = re.compile(r"\[arxiv:([\w.\-/]+?)\]", re.IGNORECASE)
TOKEN_PATTERN = re.compile(r"\b\w+\b")


def _tool_spans(trace, tool_name=None):
    for span in trace.data.spans:
        if not str(getattr(span, "span_type", "")).upper().endswith("TOOL"):
            continue
        if tool_name and getattr(span, "name", "") != tool_name:
            continue
        yield span


def _coerce(outputs):
    if isinstance(outputs, str):
        try:
            return json.loads(outputs)
        except Exception:
            return None
    return outputs


def _bleu1(reference_tokens, candidate_tokens):
    """BLEU-1: unigram precision with brevity penalty. Returns 0..1."""
    if not candidate_tokens:
        return 0.0
    ref_counts = {}
    for w in reference_tokens:
        ref_counts[w] = ref_counts.get(w, 0) + 1
    matches = 0
    for w in candidate_tokens:
        if ref_counts.get(w, 0) > 0:
            matches += 1
            ref_counts[w] -= 1
    precision = matches / len(candidate_tokens)
    if len(candidate_tokens) >= len(reference_tokens):
        bp = 1.0
    else:
        bp = math.exp(1 - len(reference_tokens) / max(1, len(candidate_tokens)))
    return bp * precision


@scorer
def citation_correctness(outputs, trace):
    """Every [arxiv:ID] in the answer must have been returned by a tool call."""
    answer = outputs if isinstance(outputs, str) else str(outputs)
    cited = set(m.group(1) for m in CITATION_PATTERN.finditer(answer))
    observed = set()
    for span in _tool_spans(trace):
        out = _coerce(span.outputs)
        items = out if isinstance(out, list) else [out] if isinstance(out, dict) else []
        for item in items:
            if isinstance(item, dict) and item.get("arxiv_id"):
                observed.add(item["arxiv_id"])
    if not cited:
        return Feedback(value=False, rationale="No [arxiv:ID] citation in the answer.")
    hallucinated = cited - observed
    if hallucinated:
        return Feedback(value=False, rationale=f"Cited IDs not seen in any tool output: {sorted(hallucinated)}.")
    return Feedback(value=True, rationale=f"All cited IDs were observed in tool outputs: {sorted(cited)}.")


@scorer
def num_results_returned(trace):
    """Total rows returned across all search_arxiv calls on this trace."""
    total = 0
    calls = 0
    for span in _tool_spans(trace, tool_name="search_arxiv"):
        calls += 1
        out = _coerce(span.outputs)
        if isinstance(out, list):
            total += len(out)
    return Feedback(value=float(total), rationale=f"{calls} search_arxiv call(s) returned {total} result(s) total.")


@scorer
def query_abstract_bleu(inputs, trace):
    """BLEU-1 between the user question (candidate) and concatenated abstracts (reference)."""
    question = inputs.get("question", "") if isinstance(inputs, dict) else str(inputs)
    abstracts = []
    for span in _tool_spans(trace):
        out = _coerce(span.outputs)
        items = out if isinstance(out, list) else [out] if isinstance(out, dict) else []
        for item in items:
            if isinstance(item, dict) and item.get("summary"):
                abstracts.append(item["summary"])
    if not abstracts:
        return Feedback(value=0.0, rationale="No abstracts in tool outputs.")
    ref_tokens = TOKEN_PATTERN.findall(" ".join(abstracts).lower())
    cand_tokens = TOKEN_PATTERN.findall(question.lower())
    score = _bleu1(ref_tokens, cand_tokens)
    return Feedback(
        value=round(float(score), 4),
        rationale=f"BLEU-1 {score:.3f} | |q|={len(cand_tokens)} |abs|={len(ref_tokens)}",
    )


citation_guidelines = Guidelines(
    name="citation_guidelines",
    guidelines=[
        "The answer must cite at least one paper using the format [arxiv:ID].",
        "The answer must not invent arxiv IDs; every cited ID must plausibly belong to a real arxiv paper.",
    ],
)

SCORERS = [
    citation_correctness,
    num_results_returned,
    query_abstract_bleu,
    citation_guidelines,
    RelevanceToQuery(),
    Safety(),
]
print("Scorers ready:", [getattr(s, "name", type(s).__name__) for s in SCORERS])

## 4. Evaluation Dataset — curated questions with ground truth

Eight seed research questions. Each record has `expected_arxiv_ids` and
`expected_facts` — **ground truth**, not rules. Rules live in the `Guidelines`
scorer above.

In [ ]:
from mlflow.genai import datasets as genai_datasets
from eval_questions import EVAL_QUESTIONS

try:
    dataset = genai_datasets.create_dataset(
        uc_table_name=DATASET_FQN,
        experiment_id=EXPERIMENT_ID,
    )
except Exception as e:
    print(f"create_dataset raised ({e}); loading existing dataset")
    dataset = genai_datasets.get_dataset(uc_table_name=DATASET_FQN)

dataset.merge_records(EVAL_QUESTIONS)
print(f"Dataset rows: {len(EVAL_QUESTIONS)} seeded into {DATASET_FQN}")

## 5. Evaluation Run v1 — expose the weakness

`mlflow.genai.evaluate()` calls the agent on every dataset row, runs all scorers, and
produces a per-row result table plus aggregate metrics logged to the run.

In [ ]:
def predict_v1(question: str) -> str:
    sid = f"eval-v1-{abs(hash(question)) % 10_000_000}"
    return run_turn(agent_v1, question, session_id=sid, user=USER_EMAIL)


with mlflow.start_run(run_name="eval-v1") as run_v1:
    result_v1 = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=predict_v1,
        scorers=SCORERS,
    )

RUN_V1_ID = run_v1.info.run_id
print(f"v1 run_id: {RUN_V1_ID}")
result_v1.result_df.head()

In [ ]:
df_v1 = result_v1.result_df
print("v1 result_df columns:", list(df_v1.columns))
fail_cols = [c for c in df_v1.columns if "citation_correctness" in c and "/value" in c]
if fail_cols:
    col = fail_cols[0]
    failing = df_v1[df_v1[col] == False]
    print(f"\nv1 citation_correctness failures: {len(failing)} / {len(df_v1)}")
    # Drop columns that hold nested Feedback / trace objects -- display() -> Arrow can't infer them
    simple_cols = [c for c in failing.columns if c not in ('assessments', 'trace')]
    print(failing[simple_cols].to_string())
else:
    print("Could not find citation_correctness column -- inspect df_v1.columns above")


## 6. Labeling Schemas — define the questions reviewers will answer

Schemas are defined **before** the session. The `type` field matters:
- `feedback` schemas collect reviewer opinions on the trace.
- `expectation` schemas collect ground truth that syncs back to the evaluation dataset.

In [ ]:
from mlflow.genai import label_schemas
from mlflow.genai.label_schemas import InputCategorical, InputText

SCHEMA_ANSWER_QUALITY = "arxiv_answer_quality"
SCHEMA_CITED_CORRECTLY = "arxiv_cited_correctly"
SCHEMA_NOTES = "arxiv_reviewer_notes"

# Schemas may already exist from prior runs; that's fine -- only create if missing.
def _ensure_schema(**kwargs):
    name = kwargs['name']
    try:
        label_schemas.create_label_schema(**kwargs)
        print(f"created schema: {name}")
    except Exception as e:
        msg = str(e)
        if 'must be unique' in msg or 'already exists' in msg or 'Duplicate' in msg:
            print(f"schema already exists: {name}")
        else:
            raise

_ensure_schema(
    name=SCHEMA_ANSWER_QUALITY,
    type="feedback",
    title="How would you rate the overall answer?",
    input=InputCategorical(options=["poor", "fair", "good", "excellent"]),
)
_ensure_schema(
    name=SCHEMA_CITED_CORRECTLY,
    type="feedback",
    title="Did the agent cite the right arXiv paper(s)?",
    input=InputCategorical(options=["yes", "no", "partial"]),
)
_ensure_schema(
    name=SCHEMA_NOTES,
    type="feedback",
    title="Reviewer notes (optional)",
    input=InputText(),
)


## 7. Labeling Session — send the failures to humans

Only label what evaluation flagged. Here we route the failing traces from v1 into a
labeling session and print the Review App URL.

In [ ]:
from mlflow.genai import labeling

session_name = f"arxiv-v1-failures-{int(time.time())}"
session = labeling.create_labeling_session(
    name=session_name,
    assigned_users=[USER_EMAIL],
    label_schemas=[SCHEMA_ANSWER_QUALITY, SCHEMA_CITED_CORRECTLY, SCHEMA_NOTES],
)

v1_traces = mlflow.search_traces(
    experiment_ids=[EXPERIMENT_ID],
    filter_string=f"run_id = '{RUN_V1_ID}'",
    return_type="list",
)
if v1_traces:
    session.add_traces(v1_traces[:5])

print(f"Labeling session: {session.name}")
print(f"Review App URL:   {session.url}")
print("Traces attached:", min(len(v1_traces), 5))

## 8. Prompts / Agent Versioning — manual v2

v2 tightens the prompt with explicit citation discipline and a tool-use plan.

In [ ]:
SYSTEM_PROMPT_V2 = (
    "You are a research assistant with access to two arXiv tools: "
    "`search_arxiv(query, max_results)` and `fetch_arxiv_paper(arxiv_id)`.\n\n"
    "How to answer every question:\n"
    "1. Call `search_arxiv` with a concise query derived from the user's question.\n"
    "2. If any result looks promising, call `fetch_arxiv_paper` for the full metadata.\n"
    "3. Write a short answer grounded in what the tools returned.\n"
    "4. Cite every paper you mention inline as `[arxiv:ID]` using the *exact* "
    "`arxiv_id` returned by a tool. Never invent or guess an arxiv id — if you did "
    "not observe it in a tool response, do not cite it.\n"
    "5. If no tool result supports the answer, say so plainly.\n"
)

mlflow.genai.register_prompt(
    name=PROMPT_FQN,
    template=SYSTEM_PROMPT_V2,
    commit_message="v2: explicit citation discipline and tool-use plan",
)
mlflow.genai.set_prompt_alias(name=PROMPT_FQN, alias="candidate", version=2)
print(f"{PROMPT_FQN} @candidate -> 2, @production still -> 1")

In [ ]:
# Tie v2 traces to a new LoggedModel
mlflow.set_active_model(name="arxiv-agent-v2")
mlflow.log_model_params({
    "prompt_fqn": PROMPT_FQN,
    "prompt_version": 2,
    "prompt_alias": "candidate",
    "llm_endpoint": AGENT_ENDPOINT,
    "tools": "search_arxiv,fetch_arxiv_paper",
})

agent_v2 = build_agent(
    endpoint=AGENT_ENDPOINT,
    system_prompt=load_prompt_by_alias(PROMPT_FQN, "candidate"),
)


def predict_v2(question: str) -> str:
    sid = f"eval-v2-{abs(hash(question)) % 10_000_000}"
    return run_turn(agent_v2, question, session_id=sid, user=USER_EMAIL)


with mlflow.start_run(run_name="eval-v2") as run_v2:
    result_v2 = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=predict_v2,
        scorers=SCORERS,
    )

RUN_V2_ID = run_v2.info.run_id
print(f"v2 run_id: {RUN_V2_ID}")
result_v2.result_df.head()


In [ ]:
# Side-by-side aggregate comparison computed from per-row scorer columns.
# Booleans -> pass rate, numerics -> mean, strings -> mode.
import pandas as pd


def _agg(result, label):
    df = result.result_df
    row = {}
    for col in df.columns:
        if not col.endswith("/value"):
            continue
        name = col.rsplit("/", 1)[0]
        series = df[col].dropna()
        if series.empty:
            continue
        if series.map(lambda x: isinstance(x, bool)).all():
            row[name] = round(float(series.astype(float).mean()), 4)
        elif pd.api.types.is_numeric_dtype(series):
            row[name] = round(float(series.mean()), 4)
        else:
            coerced = pd.to_numeric(series, errors="coerce").dropna()
            if not coerced.empty:
                row[name] = round(float(coerced.mean()), 4)
            else:
                mode = series.mode()
                row[name] = str(mode.iloc[0]) if not mode.empty else None
    return pd.Series(row, name=label)


comparison = pd.concat([_agg(result_v1, "v1"), _agg(result_v2, "v2")], axis=1)
try:
    comparison["delta"] = comparison["v2"] - comparison["v1"]
except TypeError:
    # Mixed types (strings) break arithmetic; skip delta
    pass
display(comparison.sort_index())


## 8.5 Prompt Optimization — `mlflow.genai.optimize_prompt`

Manual v2 is one way to improve the prompt. The other is to let MLflow do it:
`optimize_prompt` samples instruction variants against your dataset + scorers and
returns the one that maximizes the composite score. The expectations and scorers we
already built are the whole spec — no extra ground truth needed.

Best-effort: if the API isn't available in your MLflow build the cell skips. On
success it registers the optimized prompt as v3.

In [ ]:
v3_registered = False
try:
    import signal
    from mlflow.genai.optimize import OptimizerConfig, LLMParams

    optimizer_scorers = [citation_correctness, num_results_returned, query_abstract_bleu]

    # Hard cap (seconds) so a misbehaving optimizer can never dominate the notebook.
    OPTIMIZE_TIMEOUT_S = 600

    def _timeout_handler(signum, frame):
        raise TimeoutError(f"optimize_prompt exceeded {OPTIMIZE_TIMEOUT_S}s")

    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(OPTIMIZE_TIMEOUT_S)
    try:
        opt_result = mlflow.genai.optimize_prompt(
            target_llm_params=LLMParams(model_name=f"databricks/{AGENT_ENDPOINT}"),
            prompt=f"prompts:/{PROMPT_FQN}@candidate",
            train_data=dataset,
            scorers=optimizer_scorers,
            optimizer_config=OptimizerConfig(num_instruction_candidates=2),
        )
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

    optimized_template = opt_result.prompt.template if hasattr(opt_result, "prompt") else None
    if optimized_template:
        mlflow.genai.register_prompt(
            name=PROMPT_FQN,
            template=optimized_template,
            commit_message="v3: optimize_prompt output",
        )
        mlflow.genai.set_prompt_alias(name=PROMPT_FQN, alias="candidate", version=3)
        v3_registered = True
        print(f"Registered {PROMPT_FQN} v3 (optimize_prompt); @candidate -> 3")
        print("\nOptimized prompt preview:\n")
        print(optimized_template[:1200])
    else:
        print("optimize_prompt returned no template — skipping v3 registration.")
except ImportError as e:
    print(f"optimize_prompt not available in this MLflow build ({e}); skipping.")
except TimeoutError as e:
    print(f"optimize_prompt timed out, skipping v3: {e}")
except Exception as e:
    print(f"optimize_prompt failed, skipping v3: {type(e).__name__}: {e}")

In [ ]:
RUN_V3_ID = None
if v3_registered:
    # Tie v3 traces to its own LoggedModel
    mlflow.set_active_model(name="arxiv-agent-v3-optimized")
    mlflow.log_model_params({
        "prompt_fqn": PROMPT_FQN,
        "prompt_version": 3,
        "prompt_alias": "candidate",
        "llm_endpoint": AGENT_ENDPOINT,
        "tools": "search_arxiv,fetch_arxiv_paper",
        "source": "mlflow.genai.optimize_prompt",
    })

    agent_v3 = build_agent(
        endpoint=AGENT_ENDPOINT,
        system_prompt=load_prompt_by_alias(PROMPT_FQN, "candidate"),
    )

    def predict_v3(question: str) -> str:
        sid = f"eval-v3-{abs(hash(question)) % 10_000_000}"
        return run_turn(agent_v3, question, session_id=sid, user=USER_EMAIL)

    with mlflow.start_run(run_name="eval-v3") as run_v3:
        result_v3 = mlflow.genai.evaluate(
            data=dataset,
            predict_fn=predict_v3,
            scorers=SCORERS,
        )
    RUN_V3_ID = run_v3.info.run_id
    print(f"v3 run_id: {RUN_V3_ID}")

    three_way = pd.concat(
        [
            _agg(result_v1, "v1"),
            _agg(result_v2, "v2_manual"),
            _agg(result_v3, "v3_optimized"),
        ],
        axis=1,
    )
    display(three_way.sort_index())
else:
    print("No v3 run (optimize_prompt did not register a new version).")


## 9. Promote — move the production alias

Promote whichever version wins. One alias move, no code deploy.

In [ ]:
promote_version = 3 if v3_registered else 2
mlflow.genai.set_prompt_alias(name=PROMPT_FQN, alias="production", version=promote_version)
print(f"{PROMPT_FQN} @production -> {promote_version}")

prod_template = mlflow.genai.load_prompt(f"prompts:/{PROMPT_FQN}@production").template
print("\nProduction prompt now starts with:")
print(prod_template[:200].rstrip() + "...")

## 10. Production Monitoring — same scorers, new traces

The scorers we built for offline evaluation are the scorers we want running in
production, continuously. `register` attaches a scorer to the current experiment and
`.start(sampling_config=...)` has Databricks compute it on every new trace (or a
sample). One scorer definition, two lifecycles — dev and prod.

In [ ]:
monitor_scorers = [citation_correctness, num_results_returned, query_abstract_bleu]

try:
    from mlflow.genai.scorers import ScorerSamplingConfig
    for sc in monitor_scorers:
        sc.register(name=sc.name)
        sc.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))
        print(f"Monitoring enabled: {sc.name}")
except ImportError as e:
    print(f"ScorerSamplingConfig not available in this MLflow build ({e}); skipping.")
except Exception as e:
    print(f"scorer.register/.start failed: {type(e).__name__}: {e}")

In [ ]:
# Fire one traffic trace through @production to see the monitors light up in the UI
prod_agent = build_agent(
    endpoint=AGENT_ENDPOINT,
    system_prompt=load_prompt_by_alias(PROMPT_FQN, "production"),
)
reply = run_turn(
    prod_agent,
    "Who wrote the Direct Preference Optimization paper?",
    session_id="prod-monitor-demo",
    user=USER_EMAIL,
)
print("Production turn reply:\n", reply)

## What just happened

1. **Trace** — `mlflow.langchain.autolog()` + named `SpanType.TOOL` spans gave
   click-through visibility into every LLM and tool call.
2. **Sessions** — `run_turn` tagged each trace with `mlflow.trace.session` metadata.
3. **Judges** — `RelevanceToQuery`, `Safety`, `Guidelines` built-ins plus three
   custom scorers (`citation_correctness`, `num_results_returned`,
   `query_abstract_bleu`) that read TOOL spans.
4. **Evaluation Dataset** — UC-backed, ground-truth expectations only. Rules live in
   the Guidelines scorer.
5. **Evaluation Run** — one run per app version; aggregate comparison via
   `mlflow.get_run(...).data.metrics`.
6. **Labeling Schemas + Session** — Review App session over v1 failures.
7. **Prompts / Agent Versioning** — UC Prompt Registry with `@candidate` / `@production`
   aliases; promotion is a one-line alias move.
8. **Prompt Optimization** — `mlflow.genai.optimize_prompt` used the dataset and
   scorers to evolve a v3 prompt automatically.
9. **Production Monitoring** — the same custom scorers registered and `.start()`ed so
   they score every new production trace. Dev and prod share one eval system.

See the companion `EMAIL.md` for the opinionated, shareable summary.